In [1]:
# LSTM :Next Word Prediction

In [2]:
import pandas as pd
import numpy as np

In [4]:
data=pd.read_json('News_Category_Dataset_v3.json', lines=True)

In [5]:
data.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [6]:
# Data Exploration

In [7]:
data.shape

(209527, 6)

In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209527 entries, 0 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   link               209527 non-null  object        
 1   headline           209527 non-null  object        
 2   category           209527 non-null  object        
 3   short_description  209527 non-null  object        
 4   authors            209527 non-null  object        
 5   date               209527 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(5)
memory usage: 9.6+ MB


In [9]:
data=data['short_description'].sample(25000,random_state=42)

In [10]:
data.shape

(25000,)

In [11]:
data.isnull().sum()
data.duplicated().sum()

np.int64(2486)

In [12]:
# Data Cleaning

In [13]:
data.drop_duplicates(inplace=True)

In [14]:
data.shape

(22514,)

In [15]:
# Tokenization and Padding

In [16]:
data = data.reset_index(drop=True)

In [17]:
data[0]

"What if, in doing so, we won't just create new opportunities for ourselves, we'll also uncover ways to create new opportunities for our families that may not have otherwise existed?"

In [18]:
import tensorflow as tf

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [19]:
tokenizer=Tokenizer(oov_token='<OOV>')

In [20]:
tokenizer.fit_on_texts(data)

In [21]:
tokenizer.word_index

{'<OOV>': 1,
 'the': 2,
 'to': 3,
 'a': 4,
 'of': 5,
 'and': 6,
 'in': 7,
 'is': 8,
 'that': 9,
 'for': 10,
 'i': 11,
 'you': 12,
 'on': 13,
 'it': 14,
 'with': 15,
 'are': 16,
 'we': 17,
 'be': 18,
 'this': 19,
 'as': 20,
 'have': 21,
 'was': 22,
 'but': 23,
 'at': 24,
 'your': 25,
 'not': 26,
 'from': 27,
 'my': 28,
 'or': 29,
 'our': 30,
 'has': 31,
 'an': 32,
 'about': 33,
 'by': 34,
 'more': 35,
 'can': 36,
 'one': 37,
 'when': 38,
 'what': 39,
 'all': 40,
 'their': 41,
 'they': 42,
 'out': 43,
 'his': 44,
 'will': 45,
 "it's": 46,
 'if': 47,
 'new': 48,
 'who': 49,
 'he': 50,
 'time': 51,
 'up': 52,
 'so': 53,
 'her': 54,
 'just': 55,
 'like': 56,
 'do': 57,
 'people': 58,
 'how': 59,
 'been': 60,
 'there': 61,
 'some': 62,
 'said': 63,
 'no': 64,
 'than': 65,
 'me': 66,
 'us': 67,
 'most': 68,
 'year': 69,
 'get': 70,
 'day': 71,
 'after': 72,
 'she': 73,
 'would': 74,
 'many': 75,
 'make': 76,
 'into': 77,
 'had': 78,
 'know': 79,
 'over': 80,
 'life': 81,
 'these': 82,
 'first

In [22]:
len(tokenizer.word_index)

32785

In [23]:
data=tokenizer.texts_to_sequences(data)

In [24]:
input_sequences=[]
for seq in data:
    if len(seq)<2:
        continue
    for i in range(1,len(seq)):
        ngram=seq[:i+1]
        input_sequences.append(ngram)

In [25]:
max_len=max([len(i) for i in input_sequences])

In [26]:
max_len

167

In [27]:
input=pad_sequences(input_sequences, maxlen=max_len,padding='pre')

In [29]:
X=input[:,:-1]
y=input[:,-1]
y=np.array(y)

In [30]:
X.shape

(469940, 166)

In [31]:
y.shape

(469940,)

In [32]:
# Building the Model

In [33]:
from sklearn.model_selection import train_test_split

In [34]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [35]:
vocab_size=len(tokenizer.word_index)+1

In [36]:
max_len=X.shape[1]

In [37]:
model=Sequential()

In [38]:
model.add(Embedding(vocab_size,128,input_length=max_len))
model.add(LSTM(256))
model.add(Dense(vocab_size,activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [39]:
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [40]:
model.build(input_shape=(None,max_len))

In [41]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 166, 128)       │     4,196,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32786)          │     8,426,002 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,016,850 (49.66 MB)

 Trainable params: 13,016,850 (49.66 MB)

 Non-trainable params: 0 (0.00 B)

In [42]:
model.fit(X_train,y_train,epochs=10,validation_data=(X_test,y_test))

Epoch 1/10
11749/11749 ━━━━━━━━━━━━━━━━━━━━ 309s 26ms/step - accuracy: 0.0936 - loss: 7.0900 - val_accuracy: 0.1161 - val_loss: 6.7565
Epoch 2/10
11749/11749 ━━━━━━━━━━━━━━━━━━━━ 288s 25ms/step - accuracy: 0.1318 - loss: 6.3463 - val_accuracy: 0.1303 - val_loss: 6.6568
Epoch 3/10
11749/11749 ━━━━━━━━━━━━━━━━━━━━ 285s 24ms/step - accuracy: 0.1525 - loss: 5.9095 - val_accuracy: 0.1355 - val_loss: 6.6881
Epoch 4/10
11749/11749 ━━━━━━━━━━━━━━━━━━━━ 283s 24ms/step - accuracy: 0.1731 - loss: 5.4787 - val_accuracy: 0.1349 - val_loss: 6.8037
Epoch 5/10
11749/11749 ━━━━━━━━━━━━━━━━━━━━ 284s 24ms/step - accuracy: 0.1972 - loss: 5.0355 - val_accuracy: 0.1321 - val_loss: 6.9643
Epoch 6/10
11749/11749 ━━━━━━━━━━━━━━━━━━━━ 285s 24ms/step - accuracy: 0.2270 - loss: 4.6191 - val_accuracy: 0.1283 - val_loss: 7.1471
Epoch 7/10
11749/11749 ━━━━━━━━━━━━━━━━━━━━ 281s 24ms/step - accuracy: 0.2612 - loss: 4.2397 - val_accuracy: 0.1242 - val_loss: 7.3319
Epoch 8/10
11749/11749 ━━━━━━━━━━━━━━━━━━━━ 282s 24ms/s

In [43]:
model.save("nextword_model.h5")

In [44]:
import pickle
with open('tokenizer.pkl','wb') as file:
  pickle.dump(tokenizer,file)